In [1]:
import os

import papermill as pm
from IPython.display import Image, display

from jaxcmr.helpers import find_project_root

# LohnasKahana2014 Model_Fitting

Fit multiple CMR variants to the Lohnas & Kahana 2014 free recall dataset.

**Paradigm:** Free Recall
**Reference:** @lohnas2014retrieved

## Dataset Configuration

In [2]:
base_params = {
    "redo_fits": True,
    "redo_sims": True,
    "redo_figures": True,
    "handle_elis": False,
    "filter_repeated_recalls": False,
    "base_run_tag": "rerun",
    "experiment_count": 200,
    "max_subjects": 0,
    "base_data_tag": "LohnasKahana2014",
    "data_tag": "LohnasKahana2014",
    "data_path": "data/LohnasKahana2014.h5",
    "trial_query": "data['list_type'] > 0",
    "target_directory": "projects/repfr/results/",
    "component_paths": {
        "mfc_create_fn": "jaxcmr.components.linear_memory.init_mfc",
        "mcf_create_fn": "jaxcmr.components.linear_memory.init_mcf",
        "context_create_fn": "jaxcmr.components.context.init",
        "termination_policy_create_fn": "jaxcmr.components.termination.NoStopTermination",
    },
    "sim_alg_path": "jaxcmr.simulation.simulate_study_free_recall_and_forced_stop",
    "loss_fn_path": "jaxcmr.loss.transform_sequence_likelihood.ExcludeTerminationLikelihoodFnGenerator",
    "fit_alg_path": "jaxcmr.fitting.ScipyDE",
    "seed": 0,
    "relative_tolerance": 0.001,
    "popsize": 15,
    "num_steps": 1000,
    "cross_rate": 0.9,
    "diff_w": 0.85,
    "best_of": 1,
    "comparison_analysis_configs": [
        {"target": "jaxcmr.analyses.spc.plot_spc", "figure_suffix": "spc"},
        {"target": "jaxcmr.analyses.crp.plot_crp", "figure_suffix": "crp"},
        {"target": "jaxcmr.analyses.pnr.plot_pnr", "figure_suffix": "pnr"},
        {"target": "jaxcmr.analyses.repneighborcrp.plot_repneighborcrp_i2j", "figure_suffix": "epneighborcrp_i2j"},
        {"target": "jaxcmr.analyses.repneighborcrp.plot_repneighborcrp_j2i", "figure_suffix": "repneighborcrp_j2i"},
        {"target": "jaxcmr.analyses.repneighborcrp.plot_repneighborcrp_both", "figure_suffix": "repneighborcrp_both"},
        {"target": "jaxcmr.analyses.rpl.plot_rpl", "figure_suffix": "rpl"},
        {"target": "jaxcmr.analyses.rpl.plot_full_rpl", "figure_suffix": "full_rpl"},
    ],
    "single_analysis_configs": [
        {"target": "jaxcmr.analyses.repcrp.plot_rep_crp", "figure_suffix": "repcrp"},
        {"target": "jaxcmr.analyses.backrepcrp.plot_back_rep_crp", "figure_suffix": "backrepcrp"},
    ],
}


## Model Configurations

Core models for Chapter 3 repetition analysis:
1. **WeirdCMR** - Standard composite context model (mfc_choice_sensitivity free)
2. **WeirdPositionalCMR** - Instance-based context representation (mfc_choice_sensitivity fixed to 1.0)
3. **WeirdNoReinstateCMR** - CMR without context reinstatement
4. **WeirdCMRDistinctContexts** - CMR with separate context representations

In [3]:
varied_parameters = [
    # Baseline CMR
    {
        "redo_fits": True,
        "redo_sims": True,
        "redo_figures": True,
        "model_name": "WeirdCMRNoStop",
        "make_factory_path": "jaxcmr.models.cmr.make_factory",
        "parameters": {
            "fixed": {
                "allow_repeated_recalls": False,
                "learn_after_context_update": False,
            },
            "free": {
                "encoding_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "start_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "recall_drift_rate": [2.220446049250313e-16, 0.9999999999999998],
                "shared_support": [2.220446049250313e-16, 99.9999999999999998],
                "item_support": [2.220446049250313e-16, 99.9999999999999998],
                "learning_rate": [2.220446049250313e-16, 0.9999999999999998],
                "primacy_scale": [2.220446049250313e-16, 99.9999999999999998],
                "primacy_decay": [2.220446049250313e-16, 99.9999999999999998],
                "choice_sensitivity": [2.220446049250313e-16, 99.9999999999999998],
            },
        },
    },
]

## Run Model Fitting

In [ ]:
for partial_params in varied_parameters:

    params = base_params.copy() | {
        k: v for k, v in partial_params.items() if k != "enabled"
    }

    data_tag = params["data_tag"]
    model_name = params["model_name"]
    base_run_tag = params["base_run_tag"]
    best_of = params["best_of"]
    fit_tag = f"{base_run_tag}_best_of_{best_of}"
    max_subjects = params["max_subjects"]
    if max_subjects:
        fit_tag += f"_nsubs_{max_subjects}"

    output_path = (
        f"{find_project_root()}/projects/repfr/notebooks/rendered/"
        f"fitting_{data_tag}_{model_name}_{fit_tag}.ipynb"
    )
    print(output_path)

    pm.execute_notebook(
        f"{find_project_root()}/templates/fitting.ipynb",
        output_path,
        parameters=params,
        progress_bar=True,
    )

    for analysis_cfg in params["single_analysis_configs"]:
        figure_str = (
            f"{data_tag}_{model_name}_{fit_tag}_{analysis_cfg['figure_suffix']}.png"
        )
        figure_path = os.path.join(
            f"{find_project_root()}/projects/repfr/results/figures/fitting/{figure_str}"
        )
        print(f"![]({figure_path})")
        display(Image(filename=figure_path, width=600))

    for analysis_cfg in params["comparison_analysis_configs"]:
        figure_str = (
            f"{data_tag}_{model_name}_{fit_tag}_{analysis_cfg['figure_suffix']}.png"
        )
        figure_path = os.path.join(
            f"{find_project_root()}/projects/repfr/results/figures/fitting/{figure_str}"
        )
        print(f"![]({figure_path})")
        display(Image(filename=figure_path, width=600))


Unable to parse line 14 'trial_query = "data['listtype'] == -1" '.
Passed unknown parameter: trial_query
/Users/jordangunn/jaxcmr/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/jordangunn/jaxcmr/projects/repfr/notebooks/rendered/fitting_LohnasKahana2014_WeirdCMRNoStop_rerun_best_of_1.ipynb


Executing:  43%|████▎     | 6/14 [00:02<00:02,  3.25cell/s]